# Notebook 02 — Fine-tune Student Model

**Member responsible: B**

Runs LoRA fine-tuning for Exp 2, 3, and 4. Switch experiments by editing `configs/finetune.yaml`.

**Before running:** Make sure GPU is enabled.
Runtime > Change runtime type > T4 GPU

**Experiments:**
- Exp 2 (A1 adaptation):   set `data.mode: adaptation`
- Exp 3 (D1 distillation): set `data.mode: distillation`
- Exp 4 (D2 filtered):     set `data.mode: filtered`

**Ablations (change one field, re-run):**
- A2 rank sweep:      `lora.rank` ∈ {4, 8, 16, 32}, update `training.output_dir` each time
- A3 data size sweep: `data.train_fraction` ∈ {0.1, 0.25, 0.5, 1.0}

In [ ]:
# Step 1: Check GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Step 2: Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

# TODO: replace with your actual GitHub repo URL
!git clone https://github.com/YOUR_USERNAME/SC4001-Group133.git /content/SC4001-Group133
%cd /content/SC4001-Group133
!pip install -r requirements.txt -q

In [ ]:
# Step 3: Copy distillation datasets from Drive (needed for Exp 3, 4)
# Skip this step for Exp 2 (adaptation) — GSM8K is loaded automatically
# TODO: update the Drive path to match your folder structure
!mkdir -p data/processed
!cp -r /content/drive/MyDrive/SC4001/data/processed data/
print('Distillation datasets copied from Drive.')

In [ ]:
# Step 4: Edit configs/finetune.yaml before running
# Quick view of current config
import yaml
with open('configs/finetune.yaml') as f:
    cfg = yaml.safe_load(f)

print(f"Mode:           {cfg['data']['mode']}")
print(f"LoRA rank:      {cfg['lora']['rank']}")
print(f"Train fraction: {cfg['data']['train_fraction']}")
print(f"Output dir:     {cfg['training']['output_dir']}")
print(f"Epochs:         {cfg['training']['num_epochs']}")

In [ ]:
# Step 5: Run training
# Expected time on T4: ~2-4 hours for full dataset, 3 epochs
# Checkpoints save every 200 steps — safe to resume if session dies
from src.training.finetune import train
train()

In [ ]:
# Step 6: Save checkpoint to Google Drive
# TODO: update paths to match your output_dir and Drive structure
output_dir = cfg['training']['output_dir']
drive_path = f"/content/drive/MyDrive/SC4001/{output_dir}"
!mkdir -p "{drive_path}"
!cp -r "{output_dir}/" "{drive_path}/"
print(f'Checkpoint saved to Drive: {drive_path}')